In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q torchmetrics
!pip install -q flashlight-text
!pip install -q https://github.com/kpu/kenlm/archive/master.zip

In [ ]:
import zipfile
import os
from pathlib import Path


ZIP_ON_DRIVE = "/content/drive/MyDrive/contest3.zip"

EXTRACT_PATH = "/content/"

if os.path.exists(ZIP_ON_DRIVE):
    if not os.path.exists(os.path.join(EXTRACT_PATH, "contest3")):
        print(f"Распаковка архива {ZIP_ON_DRIVE}...")
        with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as zip_ref:
            zip_ref.extractall(EXTRACT_PATH)
        print("Распаковка успешно завершена!")
    else:
        print("Данные уже были распакованы ранее.")
else:
    print(f"Ошибка: Файл '{ZIP_ON_DRIVE}' не найден на Google Диске.")
    print("Убедитесь, что вы загрузили zip-архив на свой Google Диск и путь указан верно.")

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
from torchaudio.models.decoder import ctc_decoder
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchmetrics.text import CharErrorRate

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch  : {torch.__version__}")
print(f"torchaudio: {torchaudio.__version__}")

In [ ]:
DATASET_BASE    = Path("/content/morse_dataset_public/morse_dataset")
TRAIN_AUDIO_DIR = DATASET_BASE / "train"
TEST_AUDIO_DIR  = DATASET_BASE / "test"
LABELS_CSV      = TRAIN_AUDIO_DIR / "labels.csv"

VOCAB_CHARS = list("1234567890- ")
BLANK_IDX   = 0
CHAR2IDX    = {c: i + 1 for i, c in enumerate(VOCAB_CHARS)}
IDX2CHAR    = {v: k for k, v in CHAR2IDX.items()}
NUM_CLASSES = len(VOCAB_CHARS) + 1

print(f"Vocabulary size (incl. blank): {NUM_CLASSES}")
print(f"Chars: {VOCAB_CHARS}")

In [ ]:
SAMPLE_RATE  = 8_000
N_FFT        = 256
HOP_LENGTH   = 80
N_MELS       = 40
TOP_DB       = 80.0

mel_transform = nn.Sequential(
    T.MelSpectrogram(
        sample_rate=SAMPLE_RATE,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        power=2.0,
    ),
    T.AmplitudeToDB(stype="power", top_db=TOP_DB),
)

_dummy = torch.randn(1, SAMPLE_RATE * 5)
_feat  = mel_transform(_dummy)
print(f"Feature shape for 5 s: {_feat.shape}  →  ({N_MELS} mels × {_feat.shape[-1]} frames)")

In [ ]:
spec_augment = nn.Sequential(
    T.FrequencyMasking(freq_mask_param=4),
    T.TimeMasking(time_mask_param=5),

)


def add_noise(waveform: torch.Tensor, snr_db_range=(10, 30)) -> torch.Tensor:
    snr_db   = random.uniform(*snr_db_range)
    sig_pow  = waveform.pow(2).mean()
    noise    = torch.randn_like(waveform)
    noise_pow = noise.pow(2).mean().clamp(min=1e-9)
    scale    = (sig_pow / (noise_pow * 10 ** (snr_db / 10))).sqrt()
    return waveform + scale * noise


def time_stretch_waveform(waveform: torch.Tensor, sr: int = None,
                          rate_range=(0.85, 1.15)) -> torch.Tensor:
    rate = random.uniform(*rate_range)

    wav = waveform.unsqueeze(0)
    new_len = int(wav.shape[-1] / rate)

    stretched = torch.nn.functional.interpolate(
        wav, size=new_len, mode="linear", align_corners=False
    )

    return stretched.squeeze(0)


print("Augmentation")

In [ ]:
class MorseDataset(Dataset):
    def __init__(self,
                 audio_dir: Path,
                 labels_df: pd.DataFrame | None,
                 augment: bool = False,
                 target_sr: int = SAMPLE_RATE):
        self.audio_dir  = audio_dir
        self.augment    = augment
        self.target_sr  = target_sr

        if labels_df is not None:
            self.records = labels_df.reset_index(drop=True)
        else:
            files = sorted(audio_dir.glob("*.wav"))
            self.records = pd.DataFrame({"filename": [f.name for f in files],
                                         "text":    [""] * len(files)})

    def __len__(self) -> int:
        return len(self.records)

    def _load(self, path: Path) -> torch.Tensor:
        audio = np.fromfile(str(path), dtype="<i2", offset=44).astype(np.float32)
        audio /= 32768.0
        return torch.from_numpy(audio).unsqueeze(0)

    @staticmethod
    def _encode(label: str) -> torch.Tensor:
        return torch.tensor([CHAR2IDX[c] for c in label.upper()
                              if c.upper() in CHAR2IDX],
                            dtype=torch.long)

    def __getitem__(self, idx):
        row      = self.records.iloc[idx]
        wav_path = self.audio_dir / row["filename"]
        wav      = self._load(wav_path)

        if self.augment:
            if random.random() < 0.5:
                wav = add_noise(wav)
            if random.random() < 0.4:
                wav = time_stretch_waveform(wav, self.target_sr)

        label_idx    = self._encode(row["text"])
        label_length = len(label_idx)

        return wav.squeeze(0), label_idx, wav.shape[-1], label_length


def collate_fn(batch):
    wavs, labels, in_lens, lab_lens = zip(*batch)

    max_N  = max(w.shape[-1] for w in wavs)
    B      = len(wavs)
    padded = torch.zeros(B, max_N)
    for i, w in enumerate(wavs):
        padded[i, :w.shape[-1]] = w

    labels_cat = torch.cat(labels)
    in_lens_t  = torch.tensor(in_lens,  dtype=torch.long)
    lab_lens_t = torch.tensor(lab_lens, dtype=torch.long)

    return padded, labels_cat, in_lens_t, lab_lens_t


print("MorseDataset")

In [ ]:
df = pd.read_csv(LABELS_CSV)
print("labels.csv shape:", df.shape)
print(df.head(10))
print("\nColumn names:", df.columns.tolist())
print("\nLabel samples:")
for _, row in df.iterrows():
    print(f"  {row['filename']}  →  '{row['text']}'")

In [ ]:
if len(df) > 8:
    train_df, val_df = train_test_split(df, test_size=0.15,
                                        random_state=SEED, shuffle=True)
else:
    train_df = df.iloc[:-1].copy()
    val_df   = df.iloc[-1:].copy()

print(f"Train samples : {len(train_df)}")
print(f"Val   samples : {len(val_df)}")

BATCH_SIZE   = 128
NUM_WORKERS  = 2

train_ds = MorseDataset(TRAIN_AUDIO_DIR, train_df, augment=True)
val_ds   = MorseDataset(TRAIN_AUDIO_DIR, val_df,   augment=False)
test_ds  = MorseDataset(TEST_AUDIO_DIR,  None,     augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1,          shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS)

feats, labels_cat, in_lens, lab_lens = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  feats      : {feats.shape}")
print(f"  labels_cat : {labels_cat.shape}")
print(f"  in_lens    : {in_lens}")
print(f"  lab_lens   : {lab_lens}")

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int,
                 kernel: int = 3, freq_pool: int = 2, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel,
                      padding=kernel // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=kernel,
                      padding=kernel // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(freq_pool, 1)),
            nn.Dropout2d(p=dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MorseCTC(nn.Module):
    def __init__(self,
                 n_mels:      int   = N_MELS,
                 num_classes: int   = NUM_CLASSES,
                 cnn_channels: list = (32, 64, 128),
                 hidden_size: int   = 512,
                 num_layers:  int   = 3,
                 dropout:     float = 0.2):
        super().__init__()
        layers, in_ch = [], 1
        for out_ch in cnn_channels:
            layers.append(ConvBlock(in_ch, out_ch, freq_pool=2, dropout=0.1))
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers)
        freq_out = n_mels
        for _ in cnn_channels:
            freq_out = freq_out // 2
        rnn_input_size = cnn_channels[-1] * freq_out
        self.rnn = nn.LSTM(
            input_size=rnn_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.rnn_drop = nn.Dropout(p=dropout)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)
        x = self.cnn(x)

        B, C, F, T = x.shape
        x = x.permute(0, 3, 1, 2).reshape(B, T, C * F)

        x, _ = self.rnn(x)
        x     = self.rnn_drop(x)
        x     = self.fc(x)
        return x.permute(1, 0, 2).log_softmax(dim=-1)


model = MorseCTC().to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(model)

In [ ]:
def greedy_decode(log_probs: torch.Tensor) -> list[str]:
    indices = log_probs.argmax(dim=-1)
    indices = indices.permute(1, 0)
    results = []
    for seq in indices.cpu().tolist():
        collapsed = [seq[0]]
        for tok in seq[1:]:
            if tok != collapsed[-1]:
                collapsed.append(tok)
        text = "".join(IDX2CHAR[t] for t in collapsed if t != BLANK_IDX)
        results.append(text)
    return results


def build_beam_decoder(beam_size: int = 10):
    tokens = ["<blank>"] + VOCAB_CHARS
    decoder = ctc_decoder(
        lexicon=None,
        tokens=tokens,
        lm=None,
        nbest=1,
        beam_size=beam_size,
        blank_token="<blank>",
        sil_token=" ",
    )
    return decoder


def beam_decode(log_probs: torch.Tensor, decoder) -> list[str]:
    probs = log_probs.permute(1, 0, 2).exp().cpu()
    results_raw = decoder(probs)
    return ["".join(h[0].words) for h in results_raw]


print("Greedy decoder")

In [ ]:
ctc_loss = nn.CTCLoss(blank=BLANK_IDX, reduction="mean", zero_infinity=True)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-3,
    betas=(0.9, 0.98),
)

N_EPOCHS    = 50
scheduler   = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-6
)

scaler      = GradScaler(device=str(DEVICE))
cer_metric  = CharErrorRate()

PATIENCE         = 8
best_val_cer    = float("inf")
patience_counter = 0
best_ckpt_path   = Path("best_morse_ctc.pt")
cktp_path = Path("ckpt_morse_ctc.pt")

print(f"Optimiser : AdamW   lr={3e-4}  wd={1e-3}")
print(f"Scheduler : CosineAnnealingLR  T_max={N_EPOCHS}")
print(f"Max epochs: {N_EPOCHS}   patience: {PATIENCE}")

In [ ]:
mel_gpu = mel_transform.to(DEVICE)
spec_aug_gpu = spec_augment.to(DEVICE)

import torch.nn.functional as F

def run_epoch(loader, train, compute_cer=True):
    model.train() if train else model.eval()
    total_loss, total_cer, n_batches = 0.0, 0.0, 0

    for wavs, labels_cat, in_lens, lab_lens in loader:
        wavs       = wavs.to(DEVICE)         
        labels_cat = labels_cat.to(DEVICE)
        in_lens    = in_lens.to(DEVICE)
        lab_lens   = lab_lens.to(DEVICE)

        with torch.no_grad():
            feats = mel_gpu(wavs)           
            mean  = feats.mean(dim=(1,2), keepdim=True)
            std   = feats.std(dim=(1,2),  keepdim=True) + 1e-6
            feats = (feats - mean) / std

        if train:
            feats = spec_aug_gpu(feats)     

        in_lens_frames = in_lens // HOP_LENGTH + 1

        if train:
            optimizer.zero_grad()

        with autocast(device_type="cuda"):
            log_probs = model(feats)

            if train:
                blank_penalty = 1.5 

                log_probs_penalized = log_probs.clone()

                log_probs_penalized[..., BLANK_IDX] -= blank_penalty

                log_probs_for_loss = F.log_softmax(log_probs_penalized.float(), dim=-1)
            else:
                log_probs_for_loss = log_probs.float()

            loss = ctc_loss(log_probs_for_loss, labels_cat, in_lens_frames, lab_lens)

        if train:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()

        if compute_cer:
            with torch.no_grad():
                preds = greedy_decode(log_probs.detach())
                targets, offset = [], 0
                for length in lab_lens.tolist():
                    seg = labels_cat[offset: offset + length].tolist()
                    targets.append("".join(IDX2CHAR.get(i, "") for i in seg))
                    offset += length
                total_cer += cer_metric(preds, targets).item()

        total_loss += loss.item()
        n_batches  += 1

    return total_loss / max(n_batches, 1), total_cer / max(n_batches, 1)



print("run_epoch()")

In [ ]:
import time

model.train()
loader_iter = iter(train_loader)

t0 = time.time()
wavs, labels_cat, in_lens, lab_lens = next(loader_iter)
t1 = time.time()
print(f"Data loading : {t1-t0:.3f}s")

wavs = wavs.to(DEVICE)
labels_cat = labels_cat.to(DEVICE)
in_lens = in_lens.to(DEVICE)
lab_lens = lab_lens.to(DEVICE)

with torch.no_grad():
    feats = mel_gpu(wavs)
    mean  = feats.mean(dim=(1,2), keepdim=True)
    std   = feats.std(dim=(1,2),  keepdim=True) + 1e-6
    feats = (feats - mean) / std

in_lens_frames = (in_lens - N_FFT) // HOP_LENGTH + 1

optimizer.zero_grad()
t2 = time.time()
with autocast(device_type="cuda"):
    log_probs = model(feats)
    loss = ctc_loss(log_probs, labels_cat, in_lens_frames, lab_lens)
t3 = time.time()
print(f"Forward      : {t3-t2:.3f}s")

loss.backward()
t4 = time.time()
print(f"Backward     : {t4-t3:.3f}s")

print(f"feats shape  : {feats.shape}")
print(f"Max T        : {feats.shape[-1]}")

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_cer": [], "val_cer": []}

print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Val Loss':>10}  "
      f"{'Train CER':>10}  {'Val CER':>9}  {'LR':>9}")
print("-" * 65)

for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_cer = run_epoch(train_loader, train=True)
    vl_loss, vl_cer = run_epoch(val_loader,   train=False)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_cer"].append(tr_cer)
    history["val_cer"].append(vl_cer)

    lr_now = optimizer.param_groups[0]["lr"]
    print(f"{epoch:5d}  {tr_loss:10.4f}  {vl_loss:10.4f}  "
          f"{tr_cer:10.4f}  {vl_cer:9.4f}  {lr_now:.2e}")

    if vl_cer < best_val_cer:
        best_val_cer = vl_cer
        patience_counter = 0
        torch.save({
            "epoch":       epoch,
            "state_dict":  model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "val_cer":    best_val_cer,
        }, best_ckpt_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs).")
            break
    torch.save({
            "epoch":       epoch,
            "state_dict":  model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "val_cer":    best_val_cer,
        }, cktp_path)
print(f"\nBest val loss: {best_val_cer:.4f}  →  checkpoint saved to '{best_ckpt_path}'")

## 9. Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("CTC Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history["train_cer"], label="train")
axes[1].plot(history["val_cer"],   label="val")
axes[1].set_title("Character Error Rate")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()

In [ ]:
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  "
      f"(val_cer={ckpt['val_cer']:.4f})")

model.eval()
val_results = []
with torch.no_grad():
    for wavs, labels_cat, in_lens, lab_lens in val_loader:
        wavs = wavs.to(DEVICE)
        labels_cat = labels_cat.to(DEVICE)
        in_lens = in_lens.to(DEVICE)
        lab_lens = lab_lens.to(DEVICE)

        feats = mel_gpu(wavs)
        mean = feats.mean(dim=(1, 2), keepdim=True)
        std  = feats.std(dim=(1, 2),  keepdim=True) + 1e-6
        feats = (feats - mean) / std

        in_lens_frames = in_lens // HOP_LENGTH + 1

        log_probs = model(feats)

        preds = greedy_decode(log_probs)

        targets, offset = [], 0
        for length in lab_lens.tolist():
            seg = labels_cat[offset: offset + length].tolist()
            targets.append("".join(IDX2CHAR.get(i, "") for i in seg))
            offset += length

        for pred, gt in zip(preds, targets):
            val_results.append({"ground_truth": gt, "prediction": pred})
            print(f"   GT  : {gt!r}")
            print(f"   PRED: {pred!r}")
            print()

val_df_results = pd.DataFrame(val_results)
final_cer = cer_metric(
    val_df_results["prediction"].tolist(),
    val_df_results["ground_truth"].tolist(),
).item()
print(f"Validation CER: {final_cer:.4f}  ({final_cer*100:.1f}%)")

In [ ]:
import pandas as pd


model.eval()

greedy_results = []

print("Генерация сабмита")
with torch.no_grad():
    for idx, (wavs, _, in_lens, _) in enumerate(test_loader):
        wavs = wavs.to(DEVICE)

        feats = mel_gpu(wavs)
        mean = feats.mean(dim=(1, 2), keepdim=True)
        std  = feats.std(dim=(1, 2),  keepdim=True) + 1e-6
        feats = (feats - mean) / std

        log_probs = model(feats)

        greedy_pred = greedy_decode(log_probs)[0]

        filename = test_ds.records.iloc[idx]["filename"]

        greedy_results.append({"filename": filename, "text": greedy_pred})


df_beam = pd.DataFrame(beam_results)
df_beam.to_csv("submission_beam.csv", index=False)
print("Файл 'submission_beam.csv' успешно создан!")